# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoro-369/flyrank-ml-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My project belongs to the **Ranking / Scoring** category.

The goal is not simply to classify pages as "good" or "bad." Instead, I want to rank pages according to their priority for content review. Since content teams have limited time, they need a prioritized list of pages rather than a binary prediction.

Each page receives a score representing how strongly the observable search and content signals suggest that it should be reviewed. Higher-scoring pages appear earlier in the review queue.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

The starter dataset does not contain future outcomes, so I will use a proxy label.

The target is:

**is_declining_label = (trend_direction == "down")**

This label is derived from the observed trend direction in the current dataset. It is a proxy rather than a perfect future outcome because it represents the current state instead of future performance.

In a larger warehouse dataset, I would prefer a future-looking label such as "traffic declines during the next 30 days" to avoid leakage and better represent the real business problem.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"]
      .str.lower()
      .eq("down")
      .astype(int)
)

print(df[["trend_direction","is_declining_label"]].head())


Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.
  trend_direction  is_declining_label
0            down                   1
1            down                   1
2            down                   1
3          stable                   0
4            down                   1


## 3. Success metric

The primary evaluation metric is **Precision@50**.

Content teams typically review only a limited number of pages. Precision@50 measures how many of the top 50 recommended pages are actually declining.

This metric directly reflects the usefulness of the ranked review queue. A higher Precision@50 means editors spend more of their time reviewing pages that genuinely deserve attention.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
declining_rate = df["is_declining_label"].mean()

print(f"Declining Page Rate: {declining_rate:.2%}")
print("\nPrecision@50 will be used to evaluate the ranked recommendations.")


Declining Page Rate: 54.21%

Precision@50 will be used to evaluate the ranked recommendations.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one content page**.

Each row in the dataset represents one page and contains observable search, content, and engagement signals. These signals are used to estimate the priority of reviewing that page.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
columns = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days",
    "trend_direction"
]

print("One row = One Content Page\n")

display(df[columns].head())

print("\nNumber of pages:", len(df))
print("Number of columns:", len(df.columns))


One row = One Content Page



,content_id,impressions_90d,ctr,avg_position,word_count,content_age_days,trend_direction
0,content_304f48230142,3803,0.76,10.6,3221.0,187,down
1,content_a1fb4e703a9e,15320,0.05,20.3,2481.0,445,down
2,content_9aa793d4d895,12581,0.09,36.5,3515.0,141,down
3,content_331d6c4de07b,11751,0.49,6.2,NaN,463,stable
4,content_d99b7a2d90ca,19140,0.13,44.0,2803.0,263,down



Number of pages: 30000
Number of columns: 45


## 5. Why ML beats a fixed rule here

A fixed rule might recommend reviewing pages that are old and receive many impressions. While this captures some useful cases, it ignores many other observable signals.

Machine learning can simultaneously learn from impressions, CTR, average position, engagement, content age, freshness, and word count. These features may interact in complex ways that are difficult to capture using manually written if/else rules.

The goal is not to replace human judgment but to produce a better-ranked review queue so that editors spend their limited time reviewing the pages with the highest observed likelihood of needing attention.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "engagement_rate"
]

print("Example features that the ML model can learn from:\n")

for feature in features:
    if feature in df.columns:
        print("-", feature)


Example features that the ML model can learn from:

- impressions_90d
- ctr
- avg_position
- content_age_days
- days_since_last_update
- word_count
- engagement_rate


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs from top to bottom without errors
- [x] No client names, URLs, or private queries are included
- [x] My claims use careful, evidence-based language
- [x] Notebook committed under work/notebooks/